
<div style="background-color:rgb(100, 110, 220); padding:10px;">

## Day 6: Ensemble Methods and Hyperparameter Tuning


- **Data Preprocessing**
  - **Data Imbalance**
    - Handling imbalanced datasets (undersampling, oversampling)
    
- **Machine Learning Algorithms**
  - Introduction to Ensemble Learning (Random Forest, Gradient Boosting).
  - Implementing Stacking and Voting Classifiers
- **Model Selection and Evaluation**
  - Performing Grid Search, Random Search, and baysian for hyperparameter tuning.



### Dataset:  greenhouse.csv , geochem.csv
### libraries: pandas, scikit learn, matplotlib, skopt, and xgboost



# 1. Data Preprocessing – Handling Imbalanced Datasets

- In many real-world datasets, especially classification problems, we encounter imbalanced datasets where one class significantly outnumbers the other. For example, in fraud detection or medical diagnosis, the "positive" class (fraud, disease) may be much smaller compared to the "negative" class.

-  A model can get high accuracy by mostly predicting the majority class, yet be useless for the task that matters.

- For imbalanced data, accuracy is misleading. We care more about Recall (catching positives), Precision (avoiding false alarms), F1, and Average Precision (AP). ROC AUC is useful for overall ranking, but PR/AP is often more informative when positives are rare.


## Techniques to Handle Imbalanced Datasets
### 1. Choose an appropriate split
Use stratification for an IID classification split when class counts permit it. For temporal, grouped or spatial data, preserve the relevant boundaries first.

### 2. Class weights
Weights change the relative cost of errors; they do not guarantee better precision or recall.
- `LogisticRegression`, `SVC`, `DecisionTreeClassifier` and `RandomForestClassifier` support `class_weight='balanced'`.
- XGBoost's `scale_pos_weight` is a binary-classification option. Check each library's multiclass weighting interface separately.

```python
rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)

```


### 3. Undersampling the Majority Class
Undersampling removes some examples from the majority class to balance it with the minority class.
RandomUnderSampler; TomekLinks / EditedNearestNeighbours (ENN) / RepeatedENN; NearMiss

   - Advantages: Makes the dataset smaller and faster to train.
   - Disadvantages: You lose a significant amount of potentially useful data from the majority class.

```python
from imblearn.under_sampling import RandomUnderSampler

undersampler = RandomUnderSampler(random_state=42)
X_res, y_res = undersampler.fit_resample(X_train, y_train)


### 4. Oversampling the Minority Class
Oversampling involves adding copies of the minority class to balance the dataset. **SMOTE**, **ADASYN**, **Borderline-SMOTE**, and **Random Oversampling Examples (ROSE)** are the most common methos. There is a debate in the literature regarding the benefits of oversampling methods, as they can introduce noise, reduce interpretability, increase complexity, and lead to overfitting. Therefore, it’s often recommended to avoid these methods if there is other ways to address imbalance.

### 5. Threshold Moving (Decision Policy)

Keep the model; change the decision threshold to hit business goals (e.g., Recall ≥ 0.90 or maximize F1).

Choose the threshold via CV on training (e.g., out-of-fold scores), then fix it and report test once.

### Use Pipelines for Handling Imbalanced Data
Use an `imblearn.pipeline.Pipeline` when including a sampler. During cross-validation, both learned preprocessing and resampling must be fitted inside each training fold. Validation and test data keep their original class distribution.

# 2. Machine Learning Algorithms: Ensemble Methods

Ensembles combine multiple models. They can improve prediction when members are useful and their errors are not too strongly correlated. Improvement over the strongest individual model is not guaranteed.

## Parallel Methods:
- **Bagging (Bootstrap Aggregating):**
  - **Example:** Random Forest, Bagging can be implemented on any algorithm using `BaggingRegressor` and `BaggingClassifier` from the `sklearn.ensemble` module.

  - **How it works:** Multiple models (e.g., decision trees) are trained on different subsets of the data, and their predictions are averaged (for regression) or voted (for classification).

 
 <img src="../imgs/Bagging.jpg" alt="Description" style="width: 1400px;"/>


## Sequential Methods:
- **Boosting:**
  - **Examples:** AdaBoost, Gradient Boosting Machines (GBM), XGBoost
  - **How it works:** Models are trained one after another, with each new model focusing on the errors made by the previous models. The final prediction is a weighted sum of the predictions from all models.

   <img src="../imgs/boost.jpg" alt="Description" style="width: 1400px;"/>


## Stacking (Stacked Generalization):
- **Type:** Hybrid (both parallel and sequential elements)
- **How it works:** Multiple base models are trained in parallel on the original dataset. Their predictions are then used as input features for a meta-model, which is trained sequentially to make the final prediction.

 <img src="../imgs/stack.jpg" alt="Description" style="width: 1400px;"/>


In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

greenhouse_data = pd.read_csv('../Lecture_data/greenhouse.csv')

reg_X = greenhouse_data[['Overall Height', 'Glazing Area']]
reg_y = greenhouse_data['Average Temperature']
X_train, X_test, y_train, y_test = train_test_split(reg_X, reg_y, test_size=0.3, random_state=42)

geochem_data = pd.read_csv('../Lecture_data/geochem.csv')
Xclass = geochem_data.drop(columns='pH')
yclass = geochem_data['pH']
X_train_class, X_test_class, y_train_class, y_test_class = train_test_split(Xclass, yclass, test_size=0.3, random_state=42)


### Examples on Bagging -- Randomforest

 <img src="../imgs/RF.jpg" alt="Description" style="width: 1400px;"/>


In [3]:
## Randomforest as regressor

from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(n_estimators=100, random_state=42)
cv_results = cross_validate(forest, X_train, y_train, scoring = "neg_mean_squared_error", cv=10)

print(cv_results['test_score'])
print('mean mse: ',-1*cv_results['test_score'].mean())
print('std mse: ', cv_results['test_score'].std())

[ -8.89617401  -8.43704347 -16.0903432  -14.10370267 -14.51465231
 -11.8391712  -13.81115108  -8.34856029 -16.73266698  -8.03694764]
mean mse:  12.081041283943511
std mse:  3.232425304888495


In [4]:
## Baggingregressor

from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

bagged_model = BaggingRegressor(linear_model, n_estimators=50, random_state=42)

cv_results = cross_validate(bagged_model, X_train, y_train, scoring = "neg_mean_squared_error", cv=10)
print('mean mse: ',-1*cv_results['test_score'].mean())
print('std dev: ', cv_results['test_score'].std())

mean mse:  12.903543617745635
std dev:  3.778719775040753


In [5]:
## Voting classifier

from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

forest = RandomForestClassifier(max_depth=2, random_state=42)
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))

ensemble = VotingClassifier(
    estimators = [("rf", forest),("lr", logreg)],
    voting = 'soft', # to use predict_proba of each classifier before voting
    weights = [1,1] # to equally weight forest and logreg in the vote
)
ensemble.fit(X_train_class.values, y_train_class.values)
print(f'Test Accuracy: {ensemble.score(X_test_class.values, y_test_class.values)}')

Test Accuracy: 0.7182539682539683


### Examples on Boosting

In [6]:
from sklearn.ensemble import AdaBoostRegressor

adaboost = AdaBoostRegressor(n_estimators=50, random_state=42)
cv_results = cross_validate(adaboost, X_train, y_train, scoring = "neg_mean_squared_error", cv=10)

print('mean mse: ',-1*cv_results['test_score'].mean())
print('std dev: ', cv_results['test_score'].std())


mean mse:  12.243666036808749
std dev:  3.2258716685593902


In [7]:
from sklearn.ensemble import GradientBoostingRegressor
model = GradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, random_state=42)
scores = cross_validate(model, X_train, y_train, cv=5,
                        scoring='neg_mean_squared_error')
print(f"Gradient Boosting CV MSE: {-scores['test_score'].mean():.3f}")
# Regressor.score returns R², not classification accuracy.

Gradient Boosting CV MSE: 11.919


In [8]:
from xgboost import XGBRegressor

# Early stopping is model selection: reserve validation data from training.
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42)
xgb_reg = XGBRegressor(n_estimators=300, learning_rate=0.1,
    early_stopping_rounds=15, eval_metric='rmse',
    random_state=42, n_jobs=2)
xgb_reg.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)
print('Selected boosting rounds:', xgb_reg.best_iteration + 1)

Selected boosting rounds: 20


# 3. Model Selection and Evaluation: Hyperparameter Tuning – Grid Search,  Random Search, and Baysian optimisation

Hyperparameter tuning refers to the process of selecting the best combination of hyperparameters for a given model to maximize performance. Two popular techniques for tuning are Grid Search and Random Search.



## 3.1. Grid Search

Grid Search exhaustively searches over a predefined grid of hyperparameters and evaluates the model using cross-validation for each combination of hyperparameters.

1. **Define a set of hyperparameters and their possible values.**
2. **Train the model for every combination of hyperparameters -- never use test set for model tuning!**
3. **Use cross-validation to evaluate the performance of each combination.**
4. **Select the combination with the best mean validation performance.**

For a short classroom run, the examples below use compact search budgets. The algorithms are unchanged; a larger search explores more candidates but costs more computation.

In [9]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(random_state=42)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [3, 10],
    'min_samples_split': [2, 5]
}
grid_search = GridSearchCV(rf_model, param_grid, cv=3,
                          scoring='neg_mean_squared_error', n_jobs=2)
grid_search.fit(X_train, y_train)
print('Best parameters:', grid_search.best_params_)
print('Best CV MSE:', -grid_search.best_score_)
best_rf_model = grid_search.best_estimator_

Best parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 100}
Best CV MSE: 12.362580160797707


## 3.2. Random Search

Random Search is similar to Grid Search but instead of trying every possible combination, it randomly samples a fixed number of hyperparameter combinations from the search space.


In [10]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor

xgb_model = XGBRegressor(random_state=42, n_jobs=2)

param_dist = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 5, 10],
    'subsample': [0.6, 0.8, 1.0]
}

random_search = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_dist, n_iter=6, cv=3, random_state=42, scoring='neg_mean_squared_error')
random_search.fit(X_train, y_train)

print(f"Best Parameters: {random_search.best_params_}")

best_xgb_model = random_search.best_estimator_
print('Best CV MSE:', -random_search.best_score_)

Best Parameters: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Best CV MSE: 12.3867529856672


## 3.3 Hyperparameter Tuning with Pipelines

You can combine `GridSearchCV` or `RandomizedSearchCV` with a pipeline that includes both preprocessing and model training. This ensures that all steps are consistently applied during hyperparameter tuning.


```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42))
])
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [10, 20],
    'classifier__min_samples_split': [2, 5]
}
grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='accuracy')
grid_search.fit(X_train_class, y_train_class)
```

Scaling is unnecessary for this forest, but the example shows where a learned preprocessing step belongs when it is needed.

## 3.4 Bayesian Optimization?

Bayesian Optimization is a method to optimize hyperparameters by building a probabilistic model of the objective function and using it to select the most promising hyperparameter combinations. Unlike Grid Search and Random Search, which are more brute-force, Bayesian Optimization tries to intelligently explore and exploit the search space to find the best parameters efficiently.

The `skopt` library (scikit-optimize) provides a lightweight interface to perform Bayesian Optimization for hyperparameter tuning.


### Bayesian Optimization

#### Key Steps

1. **Define a probabilistic model** (usually Gaussian Processes).
2. **Use an acquisition function** to select the next set of hyperparameters to evaluate based on a balance of exploration and exploitation.
3. **Iteratively update the model** with new evaluations of the objective function and search for better hyperparameters.
4. **Stop** when a predefined budget (number of iterations) is exhausted or performance has converged.

The `skopt` package's `BayesSearchCV` can be used just like `GridSearchCV` and `RandomizedSearchCV`.


In [10]:
from skopt import BayesSearchCV
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

xgb_model = XGBRegressor(random_state=42, n_jobs=2)

param_space = {
    'n_estimators': (50, 200),  
    'learning_rate': (0.01, 0.2, 'log-uniform'),  
    'max_depth': (3, 10),
    'subsample': (0.6, 1.0, 'uniform') 
}

bayes_search = BayesSearchCV(
    estimator=xgb_model,
    search_spaces=param_space,
    n_iter=12,  
    cv=3,  
    scoring='neg_mean_squared_error',
    random_state=42
)

bayes_search.fit(X_train, y_train)

print(f"Best Parameters: {bayes_search.best_params_}")
best_xgb_model = bayes_search.best_estimator_
print('Best CV MSE:', -bayes_search.best_score_)

# Select using CV only, then evaluate the selected model once on the test set.
from sklearn.metrics import mean_squared_error, r2_score
searches = {'grid': grid_search, 'random': random_search, 'bayesian': bayes_search}
selected_name = max(searches, key=lambda name: searches[name].best_score_)
final_model = searches[selected_name].best_estimator_
final_pred = final_model.predict(X_test)
print('Selected search:', selected_name)
print('Final test MSE:', mean_squared_error(y_test, final_pred))
print('Final test R²:', r2_score(y_test, final_pred))

Best Parameters: OrderedDict([('learning_rate', 0.114012258603381), ('max_depth', 4), ('n_estimators', 140), ('subsample', 0.9211059124625243)])


# FAQ: 


<div style="background-color:rgba(75, 0, 130, 0.3); padding:10px;">


**1. Choosing Between Grid Search, Random Search, and Bayesian Optimization**

| Technique             | When to Use                                                                                                                                                                                                 |
|-----------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Grid Search**       | When you have a smaller number of hyperparameters to tune and want to perform an exhaustive search. It’s ideal when computational resources are not a major constraint, and the search space is relatively limited. |
| **Random Search**     | When the hyperparameter search space is large or complex, and you want to explore a wide range of values without testing every combination. It is more efficient for larger spaces and can still produce good results with fewer evaluations than Grid Search. |
| **Bayesian Optimization** | When evaluating the model is expensive (e.g., long training times or complex models) and you want to intelligently explore the search space. It is often useful for expensive objectives with a modest number of influential hyperparameters; high-dimensional or strongly categorical spaces can be challenging for a Gaussian-process surrogate. It balances exploration and exploitation by learning from previous evaluations. |


**2. Explain more on Baysian method**

## The Bayesian Method (Bayesian Inference)

The Bayesian method is based on Bayesian probability theory, which revolves around updating beliefs about the world based on new evidence. This is captured by Bayes’ Theorem, which provides a mathematical rule for how to update probabilities when new information is available.

### Bayes’ Theorem

Bayes' Theorem is expressed as:

$$
P(\theta \mid D) = \frac{P(D \mid \theta) P(\theta)}{P(D)}
$$

Where:

- $ P(\theta \mid D) $ is the posterior probability, which represents our updated belief about the parameter $\theta $ after seeing the data $ D $.
- $ P(D \mid \theta) $ is the likelihood, the probability of observing the data $ D $ given the parameter $ \theta $.
- $ P(\theta) $ is the prior probability, representing our initial belief about the parameter $ \theta $ before seeing the data.
- $ P(D) $ is the marginal likelihood or evidence, which is the probability of the data regardless of the parameters (a normalization constant).

### Core Idea Behind Bayesian Inference

1. **Prior Belief**: Start with an initial belief or assumption about the parameters of the system (this is called the prior).
2. **New Data**: Collect data or evidence.
3. **Update Belief**: Use Bayes’ Theorem to update your belief in light of the new data. This gives you the posterior distribution, which reflects the updated probability after considering the new data.

The Bayesian method is powerful because it combines prior knowledge with new evidence in a coherent way, allowing us to continuously refine our understanding as new data becomes available.

## How Bayesian Optimization Builds on Bayesian Inference

Now that we’ve introduced the Bayesian method, let’s connect it to Bayesian Optimization.

Bayesian Optimization applies the same fundamental idea of updating beliefs based on new evidence—but here, it’s used for optimizing hyperparameters or objective functions, rather than parameters in a statistical model.

### Here’s how:

1. **Objective Function as the Unknown**:
   In Bayesian Optimization, the objective function (e.g., model performance given a set of hyperparameters) is unknown and expensive to evaluate. The goal is to find the best hyperparameters that optimize this objective function. Since we don’t know the exact form of the objective function, we build a probabilistic model of it.

2. **Surrogate model-- usually a Gaussian Process as the Prior**:
   Before evaluating any hyperparameters, we start with an initial assumption about the objective function. This is similar to the prior in Bayesian inference. In Bayesian Optimization, we often use a Gaussian Process (GP) as the surrogate model, which serves as a prior over the objective function. The GP allows us to predict the objective function values for unexplored hyperparameter combinations and gives an uncertainty measure for these predictions.

   The GP assumes that similar hyperparameter values should result in similar objective function values, and it represents this assumption probabilistically.

3. **Evaluating Hyperparameters = Collecting Data**:
   As you evaluate the objective function for different hyperparameter combinations, you obtain new data points. Each evaluation is analogous to observing new evidence in Bayesian inference. This information helps to refine our understanding of the objective function.

4. **Posterior Distribution Over Objective Function**:
   After evaluating the objective function at a few hyperparameter points, we update our belief about the function using the information gathered so far. This is similar to how Bayes’ Theorem updates the posterior based on new evidence. In the context of Bayesian Optimization, this updated belief about the objective function is captured by the GP’s posterior distribution, which incorporates both the function values we’ve observed and the uncertainty around unobserved points.

5. **Acquisition Function to Select the Next Hyperparameters**:
   The acquisition function plays the role of decision-making in Bayesian Optimization. Based on the posterior distribution of the objective function (updated by the GP after each evaluation), the acquisition function decides which hyperparameters to try next. It considers both:
   - **Exploitation**: The current best guess of where good hyperparameters might be.
   - **Exploration**: The regions of the hyperparameter space where we have high uncertainty (and thus might discover even better values).

   This step mirrors the decision-making process in Bayesian inference, where we might choose new experiments or observations that are most likely to improve our understanding of a system.
</div>
